In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor




In [2]:
BASE_DIR = "data/osic-pulmonary-fibrosis-progression"
if not os.path.exists(os.path.join(BASE_DIR, "train.csv")):
    BASE_DIR = "/kaggle/input/osic-pulmonary-fibrosis-progression"

TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")
TEST_CSV = os.path.join(BASE_DIR, "test.csv")
SAMPLE_SUBMISSION_CSV = os.path.join(BASE_DIR, "sample_submission.csv")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_CSV)


def add_typical_fvc(df):
    df = df.copy()
    df["typical_fvc"] = df["FVC"] / df["Percent"].replace(0, np.nan) * 100.0
    df["typical_fvc"].fillna(df["FVC"], inplace=True)
    return df


train_df = add_typical_fvc(train_df)
test_df = add_typical_fvc(test_df)


def encode_df(df):
    df = df.copy()
    df["Male"] = (df["Sex"] == "Male").astype(int)
    df["Female"] = (df["Sex"] == "Female").astype(int)
    df["Never_smoked"] = (df["SmokingStatus"] == "Never smoked").astype(int)
    df["Currently_smokes"] = (df["SmokingStatus"] == "Currently smokes").astype(int)
    df["Ex_smoker"] = (df["SmokingStatus"] == "Ex-smoker").astype(int)
    return df


train_df = encode_df(train_df)
test_df = encode_df(test_df)




/tmp/ipykernel_11/3696734610.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["typical_fvc"].fillna(df["FVC"], inplace=True)
/tmp/ipykernel_11/3696734610.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

In [3]:
FEATURE_COLS = [
    "Weeks",
    "Age",
    "typical_fvc",
    "Male",
    "Female",
    "Never_smoked",
    "Currently_smokes",
    "Ex_smoker",
]

X = train_df[FEATURE_COLS].values
y = train_df["FVC"].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(
    n_estimators=1500,
    max_depth=None,
    min_samples_leaf=1,
    max_features=None,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

val_pred = rf.predict(X_val)

residual_std = np.std(y_val - val_pred)

val_weeks = X_val[:, 0]  # Weeks column
errors = np.abs(y_val - val_pred)
df_err = pd.DataFrame({"Weeks": val_weeks, "error": errors})
week_std_series = df_err.groupby("Weeks")["error"].std()
week_std_dict = week_std_series.to_dict()

BASE_CONFIDENCE = max(70, residual_std)




In [4]:
baseline_lookup = {}
for _, row in test_df.iterrows():
    patient = row["Patient"]
    baseline_lookup[patient] = {
        "Age": row["Age"],
        "typical_fvc": row["typical_fvc"],
        "Male": row["Male"],
        "Female": row["Female"],
        "Never_smoked": row["Never_smoked"],
        "Currently_smokes": row["Currently_smokes"],
        "Ex_smoker": row["Ex_smoker"],
    }

submission_features = []
submission_ids = []
submission_weeks = []  # keep weeks for per‑week confidence
for pid_week in sample_submission["Patient_Week"]:
    patient_id, week_str = pid_week.rsplit("_", 1)
    week = float(week_str)
    base = baseline_lookup.get(patient_id)
    if base is None:
        base = {
            "Age": train_df["Age"].mean(),
            "typical_fvc": train_df["typical_fvc"].mean(),
            "Male": 0,
            "Female": 0,
            "Never_smoked": 0,
            "Currently_smokes": 0,
            "Ex_smoker": 0,
        }
    feat = [
        week,
        base["Age"],
        base["typical_fvc"],
        base["Male"],
        base["Female"],
        base["Never_smoked"],
        base["Currently_smokes"],
        base["Ex_smoker"],
    ]
    submission_features.append(feat)
    submission_ids.append(pid_week)
    submission_weeks.append(week)

submission_X = np.array(submission_features, dtype=np.float32)




In [5]:
pred_fvc = rf.predict(submission_X)

FVC_MIN, FVC_MAX = train_df["FVC"].min(), train_df["FVC"].max()
pred_fvc = np.clip(pred_fvc, FVC_MIN, FVC_MAX)

# Increase confidence scaling to reduce the Δ/σ penalty more aggressively
SCALING_FACTOR = 2.5
GLOBAL_CONFIDENCE = np.clip(BASE_CONFIDENCE * SCALING_FACTOR, 70, None)

confidence_vals = []
for wk in submission_weeks:
    week_std = week_std_dict.get(wk)
    if week_std is not None:
        sigma = max(70, week_std) * SCALING_FACTOR
    else:
        sigma = GLOBAL_CONFIDENCE
    confidence_vals.append(sigma)

confidence = np.array(confidence_vals, dtype=np.float32)

final_sub = pd.DataFrame(
    {
        "Patient_Week": submission_ids,
        "FVC": pred_fvc.astype(int),
        "Confidence": confidence.astype(int),
    }
)




In [6]:
output_path = "submission.csv"
final_sub.to_csv(output_path, index=False)
print(f"Submission written to {output_path}")
print(final_sub.head())

Submission written to submission.csv
                   Patient_Week   FVC  Confidence
0  ID00126637202218610655908_-3  2977         605
1  ID00126637202218610655908_-2  2977         605
2  ID00126637202218610655908_-1  2977         175
3   ID00126637202218610655908_0  2978         268
4   ID00126637202218610655908_1  2968         312
